# LLM sentiment analysis

Historical 100-review model comparison, validation workbook generation, full Gemini run and recovery cells. Run only the relevant section in a fresh kernel. Preserve prompt versions and existing checkpoints. Paid API calls are possible.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
#%pip install -U anthropic google-genai pandas numpy tqdm openpyxl

In [ ]:
import os
import json
import time
import random
import re
import hashlib
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import anthropic
from google import genai
from google.genai import types


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = HF_DIR

INPUT_FILE = DATA_DIR / "df_final_absa.csv"

OUTPUT_DIR = LLM_OUTPUT_DIR / "Pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "haiku": "claude-haiku-5-5",
    "gemini": "gemini-3.1-flash-lite",
}

# Standard API prices per million tokens, USD
PRICING = {
    "haiku": {"input": 0.10, "output": 0.50},
    "gemini": {"input": 0.25, "output": 1.50},
}

# Run both models independently.
SELECTED_MODELS = ["haiku", "gemini"]

# Start with 100 reviews.
# Set to None to process all eligible reviews.
MAX_REVIEWS = 100

RANDOM_STATE = 42

MAX_RETRIES = 5
BASE_RETRY_DELAY = 3

# Save each completed review immediately to JSONL.
# This allows a run to resume after interruption.

FEATURES = [
    "long_haul",
    "first_class",
    "cancellation",
    "delay",
    "boarding",
    "baggage",
    "economy_class",
    "business_class",
    "international",
    "domestic",
    "customer_service",
    "cabin_crew",
    "ground_staff",
    "complaint_resolution",
    "seat_comfort",
    "food_beverage",
    "cabin_cleanliness",
    "entertainment_wifi",
    "booking_experience",
    "check_in",
    "airport_experience",
    "value_for_money",
    "pricing_fees",
    "flight_disruption",
    "overall_experience",
]

SENTIMENT_LABELS = {
    "positive",
    "negative",
    "neutral",
    "mixed",
    "not_mentioned",
}

# These features describe flight characteristics rather
# than necessarily sentiment-bearing service experiences.
CONTEXT_FEATURES = {
    "long_haul",
    "first_class",
    "economy_class",
    "business_class",
    "international",
    "domestic",
}

FEATURE_DEFINITIONS = {
    "long_haul":
        "A flight lasting strictly more than six hours; "
        "do not confuse total connecting journey time with flight duration.",
    "first_class":
        "The passenger actually traveled in first class.",
    "cancellation":
        "An actual flight cancellation is discussed.",
    "delay":
        "A flight departure or arrival delay is discussed.",
    "boarding":
        "The boarding process or boarding experience.",
    "baggage":
        "Checked baggage, cabin baggage, baggage handling or allowances.",
    "economy_class":
        "The passenger actually traveled in economy class.",
    "business_class":
        "The passenger actually traveled in business class.",
    "international":
        "A flight crossing a national border.",
    "domestic":
        "A flight within one country.",
    "customer_service":
        "Customer support, telephone support or airline customer care.",
    "cabin_crew":
        "Flight attendants and cabin crew interactions.",
    "ground_staff":
        "Airline ground employees and gate or check-in agents.",
    "complaint_resolution":
        "Handling complaints, compensation claims, refunds or disputes.",
    "seat_comfort":
        "Seat comfort, legroom, seat pitch or recline.",
    "food_beverage":
        "Food, drinks, catering and onboard meals.",
    "cabin_cleanliness":
        "Cabin hygiene, cleanliness, toilets or sanitation.",
    "entertainment_wifi":
        "Inflight entertainment, connectivity or Wi-Fi.",
    "booking_experience":
        "Booking, reservation, ticket purchase or booking changes.",
    "check_in":
        "Online, mobile or airport check-in process.",
    "airport_experience":
        "Airport facilities, lounges, terminal or airport experience.",
    "value_for_money":
        "Whether the service or experience was worth its cost.",
    "pricing_fees":
        "Ticket prices, fees, charges and ancillary costs.",
    "flight_disruption":
        "Operational disruption, diversions, missed connections "
        "or involuntary rebooking.",
    "overall_experience":
        "Explicit overall judgment of the airline or journey.",
}

# ============================================================
# 2. API CREDENTIALS
# ============================================================

if "haiku" in SELECTED_MODELS:
    if not os.getenv("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass(
            "Anthropic API key: "
        )

if "gemini" in SELECTED_MODELS:
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass(
            "Gemini API key: "
        )

anthropic_client = (
    anthropic.Anthropic()
    if "haiku" in SELECTED_MODELS
    else None
)

gemini_client = (
    genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    if "gemini" in SELECTED_MODELS
    else None
)

# ============================================================
# 3. LOAD DATA
# ============================================================

df_source = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

required_columns = [
    "airline",
    "review_en",
    "long_haul",
    "first_class",
]

missing = [
    col for col in required_columns
    if col not in df_source.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Preserve the original row identity.
# Prefer an existing unique record_key when available.
if (
    "record_key" not in df_source.columns
    or df_source["record_key"].isna().any()
    or not df_source["record_key"].is_unique
):
    raise ValueError("A unique, non-null record_key is required.")

df_source["record_key"] = df_source["record_key"].astype(str)

for col in ["long_haul", "first_class"]:
    values = pd.to_numeric(
        df_source[col],
        errors="coerce"
    )

    if not values.dropna().isin([0, 1]).all():
        raise ValueError(
            f"{col} contains missing or invalid flags. "
            "Resolve these before applying the exclusion."
        )

    df_source[col] = values.fillna(0).astype(int)

df_source["review_en"] = (
    df_source["review_en"]
    .fillna("")
    .astype(str)
)

# Exclude long-haul and first-class reviews.
df_work = df_source[
    (df_source["long_haul"].fillna(0) != 1)
    & (df_source["first_class"].fillna(0) != 1)
    & (df_source["review_en"].str.strip() != "")
    & pd.to_datetime(df_source["review_date"], errors="coerce", utc=True).ge(pd.Timestamp("2024-01-01", tz="UTC"))
].copy()

# Stable ordering makes resuming predictable.
df_work = df_work.sort_values(
    "record_key"
).reset_index(drop=True)

# Select the same sample for both models.
if MAX_REVIEWS is not None:
    df_work = (
        df_work.sample(
            n=min(MAX_REVIEWS, len(df_work)),
            random_state=RANDOM_STATE
        )
        .sort_values("record_key")
        .reset_index(drop=True)
    )

print("Original records:", len(df_source))
print("Eligible non-empty reviews:", len(df_work))
print("Models:", SELECTED_MODELS)

In [ ]:
# ============================================================
# 4. COMMON CLASSIFICATION PROMPT
# ============================================================

FEATURE_GUIDE = "\n".join(
    f"- {feature}: {definition}"
    for feature, definition in FEATURE_DEFINITIONS.items()
)

SYSTEM_PROMPT = f"""
You are an expert airline passenger-review annotator.

Your task is to classify airline reviews using exactly
25 predefined features and to perform aspect-based
sentiment analysis.

FEATURE DEFINITIONS:
{FEATURE_GUIDE}

CLASSIFICATION RULES:

1. Return every one of the 25 features.
2. For each feature, return:
   - classification: integer 0 or 1
   - sentiment: one of positive, negative, neutral,
     mixed, not_mentioned
3. Classification 1 means the review or supplied
   structured evidence supports the feature.
4. Classification 0 means insufficient evidence.
   It does not necessarily mean the opposite is true.
5. Do not classify cabin classes merely because the
   passenger mentions another cabin or an upgrade.
6. Do not assume a long-haul flight from total journey
   duration, layovers or the word "international".
7. Do not invent flight routes, dates or durations.
8. For contextual features such as cabin class and
   international/domestic, sentiment may be
   not_mentioned even when classification is 1.
9. For passenger-experience features:
   - positive: clearly favorable assessment
   - negative: clearly unfavorable assessment
   - neutral: aspect discussed without clear polarity
   - mixed: both favorable and unfavorable assessments
   - not_mentioned: aspect not discussed or no
     sentiment-bearing evidence
10. If classification is 0, sentiment must be
    not_mentioned.
11. A feature with classification 1 and no expressed
    sentiment should usually be neutral for an
    experience aspect, or not_mentioned for a
    contextual feature.
12. Assess sentiment toward the specific aspect,
    not toward the review as a whole.
13. Provide overall_sentiment using:
    positive, negative, neutral, mixed.
14. Do not infer passenger satisfaction from a
    numerical rating unless explicitly requested.
15. Do not let the airline name affect your judgments.
16. Return only valid JSON, without markdown fences.

OUTPUT STRUCTURE:

{{
  "overall_sentiment": "negative",
  "features": {{
    "delay": {{
      "classification": 1,
      "sentiment": "negative"
    }}
  }}
}}

The actual features object must include all 25 feature names.
Do not add explanations or extra keys.
"""


def build_user_prompt(row):
    """
    Include the review and optional structured metadata.
    Do not include existing rule-based or Hugging Face
    predictions: the LLM assessment must be independent.
    """

    metadata_columns = [
        "origin",
        "destination",
        "route",
        "flight_number",
        "travel_class",
    ]

    metadata = {}

    for col in metadata_columns:
        if col in row.index and pd.notna(row[col]):
            value = str(row[col]).strip()

            if value:
                metadata[col] = value

    payload = {
        "review": str(row["review_en"]),
        "metadata": metadata,
    }

    return json.dumps(
        payload,
        ensure_ascii=False
    )


def extract_json(text):
    """
    Parse the model's JSON response.
    Accepts an optional markdown code fence.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text
        )
        text = re.sub(
            r"\s*```$",
            "",
            text
        )

    return json.loads(text)


def validate_result(result):
    """
    Reject incomplete or inconsistent predictions.
    Never silently fill missing features with zeros.
    """

    if not isinstance(result, dict):
        raise ValueError("Result must be a JSON object")

    overall = result.get("overall_sentiment")

    if overall not in (
        SENTIMENT_LABELS - {"not_mentioned"}
    ):
        raise ValueError(
            f"Invalid overall sentiment: {overall}"
        )

    features = result.get("features")

    if not isinstance(features, dict):
        raise ValueError("Missing features object")

    if set(features) != set(FEATURES):
        missing = set(FEATURES) - set(features)
        extra = set(features) - set(FEATURES)

        raise ValueError(
            f"Feature mismatch. Missing={missing}; "
            f"extra={extra}"
        )

    for feature in FEATURES:
        item = features[feature]

        if not isinstance(item, dict):
            raise ValueError(f"Invalid {feature} object")

        classification = item.get("classification")
        sentiment = item.get("sentiment")

        if type(classification) is not int:
            raise ValueError(
                f"{feature}: classification must be integer"
            )

        if classification not in (0, 1):
            raise ValueError(
                f"{feature}: invalid classification"
            )

        if sentiment not in SENTIMENT_LABELS:
            raise ValueError(
                f"{feature}: invalid sentiment {sentiment}"
            )

        if (
            classification == 0
            and sentiment != "not_mentioned"
        ):
            raise ValueError(
                f"{feature}: absent feature has sentiment"
            )

    return result

In [ ]:
# ============================================================
# 5. MODEL INFERENCE
# ============================================================

def call_haiku(user_prompt):

    """Return or process call haiku."""
    response = anthropic_client.messages.create(
        model=MODELS["haiku"],
        max_tokens=2200,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
    )

    response_text = "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )

    if response.stop_reason == "max_tokens":
        raise ValueError(
            "Haiku response truncated. Increase max_tokens."
        )

    result = validate_result(
        extract_json(response_text)
    )

    return {
        "result": result,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def call_gemini(user_prompt):

    """Return or process call gemini."""
    response = gemini_client.models.generate_content(
        model=MODELS["gemini"],
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            max_output_tokens=2200,
            # Keep the classification task lightweight.
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            ),
        ),
    )

    if not response.text:
        raise ValueError(
            "Gemini returned no text"
        )

    result = validate_result(
        extract_json(response.text)
    )

    usage = response.usage_metadata

    input_tokens = (
        usage.prompt_token_count or 0
        if usage else 0
    )

    # Gemini's billed output can include thinking tokens.
    output_tokens = (
        (usage.candidates_token_count or 0)
        + (usage.thoughts_token_count or 0)
        if usage else 0
    )

    return {
        "result": result,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }


MODEL_FUNCTIONS = {
    "haiku": call_haiku,
    "gemini": call_gemini,
}


def calculate_cost(model_key, input_tokens, output_tokens):

    """Return or process calculate cost."""
    prices = PRICING[model_key]

    return (
        input_tokens * prices["input"]
        + output_tokens * prices["output"]
    ) / 1_000_000

In [ ]:
# ============================================================
# 6. CHECKPOINTING AND EXECUTION
# ============================================================

def prompt_hash():
    """Return or process prompt hash."""
    return hashlib.sha256(
        SYSTEM_PROMPT.encode("utf-8")
    ).hexdigest()[:16]


def review_hash(user_prompt):
    """Return or process review hash."""
    return hashlib.sha256(
        user_prompt.encode("utf-8")
    ).hexdigest()


def load_checkpoint(path):
    """
    Return previously completed predictions keyed by
    record_key. A changed prompt or review is reprocessed.
    """

    records = {}

    if not path.exists():
        return records

    with path.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:
            if not line.strip():
                continue

            try:
                item = json.loads(line)

                if item.get("status") == "success":
                    records[str(item["record_key"])] = item

            except (ValueError, KeyError):
                continue

    return records


def call_with_retries(model_key, user_prompt):

    """Return or process call with retries."""
    function = MODEL_FUNCTIONS[model_key]

    for attempt in range(MAX_RETRIES):

        try:
            return function(user_prompt)

        except Exception as exc:

            if attempt == MAX_RETRIES - 1:
                raise

            delay = (
                BASE_RETRY_DELAY * (2 ** attempt)
                + random.uniform(0, 2)
            )

            print(
                f"{model_key}: attempt {attempt + 1} "
                f"failed: {str(exc)[:150]}. "
                f"Retrying in {delay:.1f}s"
            )

            time.sleep(delay)


def run_model(model_key, dataframe):

    """Return or process run model."""
    checkpoint_path = (
        OUTPUT_DIR / f"{model_key}_checkpoint.jsonl"
    )

    failures_path = (
        OUTPUT_DIR / f"{model_key}_failures.jsonl"
    )

    completed = load_checkpoint(checkpoint_path)

    print(
        f"\nRunning {model_key}: "
        f"{len(dataframe):,} reviews"
    )

    print(
        f"Existing successful checkpoint records: "
        f"{len(completed):,}"
    )

    successes = 0
    failures = 0
    skipped = 0

    with checkpoint_path.open(
        "a",
        encoding="utf-8"
    ) as checkpoint_file, failures_path.open(
        "a",
        encoding="utf-8"
    ) as failure_file:

        for _, row in tqdm(
            dataframe.iterrows(),
            total=len(dataframe),
            desc=model_key
        ):

            record_key = str(row["record_key"])

            user_prompt = build_user_prompt(row)
            current_review_hash = review_hash(user_prompt)

            previous = completed.get(record_key)

            if (
                previous
                and previous.get("prompt_hash") == prompt_hash()
                and previous.get("review_hash") == current_review_hash
                and previous.get("model_id") == MODELS[model_key]
            ):
                skipped += 1
                continue

            started = time.perf_counter()

            try:

                output = call_with_retries(
                    model_key,
                    user_prompt
                )

                elapsed = (
                    time.perf_counter() - started
                )

                cost = calculate_cost(
                    model_key,
                    output["input_tokens"],
                    output["output_tokens"]
                )

                item = {
                    "record_key": record_key,
                    "model": model_key,
                    "model_id": MODELS[model_key],
                    "prompt_hash": prompt_hash(),
                    "review_hash": current_review_hash,
                    "status": "success",
                    "result": output["result"],
                    "input_tokens": output["input_tokens"],
                    "output_tokens": output["output_tokens"],
                    "estimated_cost_usd": cost,
                    "elapsed_seconds": elapsed,
                }

                checkpoint_file.write(
                    json.dumps(
                        item,
                        ensure_ascii=False
                    ) + "\n"
                )

                checkpoint_file.flush()

                completed[record_key] = item
                successes += 1

            except Exception as exc:

                failures += 1

                failure_file.write(
                    json.dumps({
                        "record_key": record_key,
                        "model": model_key,
                        "error": str(exc),
                    }) + "\n"
                )

                failure_file.flush()

    print(
        f"{model_key}: new successes={successes}, "
        f"skipped={skipped}, failures={failures}"
    )

    return checkpoint_path

In [ ]:
# ============================================================
# 7. EXECUTE BOTH MODELS
# ============================================================

checkpoint_paths = {}

for model_key in SELECTED_MODELS:

    checkpoint_paths[model_key] = run_model(
        model_key,
        df_work
    )

print("\nBoth model runs finished.")

In [ ]:
# ============================================================
# 8. CONVERT CHECKPOINTS TO DATAFRAMES
# ============================================================

def checkpoint_to_dataframe(model_key):

    """Return or process checkpoint to dataframe."""
    checkpoint_path = (
        OUTPUT_DIR / f"{model_key}_checkpoint.jsonl"
    )

    records = load_checkpoint(checkpoint_path)

    rows = []

    for item in records.values():

        if (
            item.get("prompt_hash") != prompt_hash()
            or item.get("model_id") != MODELS[model_key]
        ):
            continue

        result = item["result"]

        flat = {
            "record_key": item["record_key"],
            "model": model_key,
            "model_id": item["model_id"],
            "overall_sentiment": result["overall_sentiment"],
            "input_tokens": item["input_tokens"],
            "output_tokens": item["output_tokens"],
            "estimated_cost_usd": item["estimated_cost_usd"],
            "elapsed_seconds": item["elapsed_seconds"],
            "review_hash": item["review_hash"],
        }

        for feature in FEATURES:

            feature_result = result["features"][feature]

            flat[f"{feature}_llm"] = (
                feature_result["classification"]
            )

            flat[f"{feature}_sentiment_llm"] = (
                feature_result["sentiment"]
            )

        rows.append(flat)

    return pd.DataFrame(rows)


def export_model_results(model_key):

    """Return or process export model results."""
    df_predictions = checkpoint_to_dataframe(
        model_key
    )

    if df_predictions.empty:
        print(f"No successful predictions for {model_key}")
        return pd.DataFrame()

    # Only keep predictions matching the current dataset.
    current_hashes = {
        str(row["record_key"]): review_hash(
            build_user_prompt(row)
        )
        for _, row in df_work.iterrows()
    }

    df_predictions = df_predictions[
        df_predictions.apply(
            lambda row: (
                current_hashes.get(row["record_key"])
                == row["review_hash"]
            ),
            axis=1
        )
    ].copy()

    if df_predictions.empty:
        return pd.DataFrame()

    df_result = df_work.merge(
        df_predictions,
        on="record_key",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_llm_metadata")
    )

    output_file = (
        OUTPUT_DIR /
        f"df_final_absa_{model_key}.csv"
    )

    df_result.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved: {output_file}")
    print(f"Records: {len(df_result):,}")

    return df_result


df_haiku = (
    export_model_results("haiku")
    if "haiku" in SELECTED_MODELS
    else pd.DataFrame()
)

df_gemini = (
    export_model_results("gemini")
    if "gemini" in SELECTED_MODELS
    else pd.DataFrame()
)

In [ ]:
# ============================================================
# 9. COST COMPARISON
# ============================================================

cost_rows = []

for model_key in SELECTED_MODELS:

    result = checkpoint_to_dataframe(model_key)

    if result.empty:
        continue

    # Restrict the report to the current sample.
    result = result[
        result["record_key"].isin(
            df_work["record_key"]
        )
    ].copy()

    n = len(result)

    cost_rows.append({
        "model": model_key,
        "reviews_processed": n,
        "input_tokens": result["input_tokens"].sum(),
        "output_tokens": result["output_tokens"].sum(),
        "total_cost_usd": result["estimated_cost_usd"].sum(),
        "cost_per_1000_reviews_usd": (
            result["estimated_cost_usd"].sum()
            / n * 1000
        ),
        "avg_seconds_per_review": (
            result["elapsed_seconds"].mean()
        ),
    })

df_cost_comparison = pd.DataFrame(cost_rows)

display(df_cost_comparison)

df_cost_comparison.to_csv(
    OUTPUT_DIR / "llm_cost_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# 10. MODEL AGREEMENT
# ============================================================

if not df_haiku.empty and not df_gemini.empty:

    comparison = df_haiku.merge(
        df_gemini,
        on="record_key",
        suffixes=("_haiku", "_gemini"),
        validate="one_to_one"
    )

    agreement_rows = []

    for feature in FEATURES:

        binary_h = f"{feature}_llm_haiku"
        binary_g = f"{feature}_llm_gemini"

        sentiment_h = f"{feature}_sentiment_llm_haiku"
        sentiment_g = f"{feature}_sentiment_llm_gemini"

        binary_agreement = (
            comparison[binary_h]
            == comparison[binary_g]
        ).mean()

        sentiment_agreement = (
            comparison[sentiment_h]
            == comparison[sentiment_g]
        ).mean()

        # Compare sentiment only where both models
        # classified the aspect as present.
        both_present = (
            (comparison[binary_h] == 1)
            & (comparison[binary_g] == 1)
        )

        if both_present.any():
            sentiment_agreement_present = (
                comparison.loc[both_present, sentiment_h]
                == comparison.loc[both_present, sentiment_g]
            ).mean()
        else:
            sentiment_agreement_present = np.nan

        agreement_rows.append({
            "feature": feature,
            "binary_agreement": binary_agreement,
            "sentiment_agreement_all": sentiment_agreement,
            "sentiment_agreement_both_present":
                sentiment_agreement_present,
            "n_both_present": int(both_present.sum()),
        })

    df_agreement = pd.DataFrame(agreement_rows)

    overall_agreement = (
        comparison["overall_sentiment_haiku"]
        == comparison["overall_sentiment_gemini"]
    ).mean()

    print(
        f"Overall sentiment agreement: "
        f"{overall_agreement:.2%}"
    )

    display(df_agreement)

    df_agreement.to_csv(
        OUTPUT_DIR / "llm_feature_agreement.csv",
        index=False,
        encoding="utf-8-sig"
    )

    comparison.to_csv(
        OUTPUT_DIR / "llm_full_comparison.csv",
        index=False,
        encoding="utf-8-sig"
    )

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

DATA_DIR = HF_DIR

LLM_DIR = LLM_OUTPUT_DIR

HF_FILE = DATA_DIR / "df_final_absa.csv"
HAIKU_FILE = LLM_DIR / "Pilot" / "df_final_absa_haiku.csv"
GEMINI_FILE = LLM_DIR / "Pilot" / "df_final_absa_gemini.csv"

VALIDATION_FILE = LLM_DIR / "model_validation_100_reviews.xlsx"

FEATURES = [
    "long_haul", "first_class", "cancellation",
    "delay", "boarding", "baggage",
    "economy_class", "business_class",
    "international", "domestic",
    "customer_service", "cabin_crew",
    "ground_staff", "complaint_resolution",
    "seat_comfort", "food_beverage",
    "cabin_cleanliness", "entertainment_wifi",
    "booking_experience", "check_in",
    "airport_experience", "value_for_money",
    "pricing_fees", "flight_disruption",
    "overall_experience"
]

SENTIMENTS = [
    "positive", "negative", "neutral",
    "mixed", "not_mentioned"
]

CONTEXT_FEATURES = {
    "long_haul", "first_class",
    "economy_class", "business_class",
    "international", "domestic"
}

FEATURE_DESCRIPTIONS = {
    "long_haul": "Individual flight lasting strictly more than 6 hours",
    "first_class": "Passenger actually traveled in first class",
    "cancellation": "Flight cancellation",
    "delay": "Delayed departure or arrival",
    "boarding": "Boarding process",
    "baggage": "Luggage, baggage handling or allowances",
    "economy_class": "Passenger actually traveled in economy",
    "business_class": "Passenger actually traveled in business",
    "international": "Flight crossing a national border",
    "domestic": "Flight within one country",
    "customer_service": "Airline customer support or customer care",
    "cabin_crew": "Flight attendants and onboard crew",
    "ground_staff": "Airport-based airline employees",
    "complaint_resolution": "Complaints, refunds, claims or compensation",
    "seat_comfort": "Seats, legroom, recline and comfort",
    "food_beverage": "Food, meals, drinks and catering",
    "cabin_cleanliness": "Cabin hygiene and cleanliness",
    "entertainment_wifi": "Inflight entertainment, connectivity or Wi-Fi",
    "booking_experience": "Ticket booking or reservation process",
    "check_in": "Online or airport check-in",
    "airport_experience": "Airport facilities, lounges or terminal",
    "value_for_money": "Whether the experience was worth its cost",
    "pricing_fees": "Ticket prices, charges and additional fees",
    "flight_disruption": "Diversions, missed connections or rebooking",
    "overall_experience": "Explicit overall evaluation of the airline or journey"
}

In [ ]:
df_hf = pd.read_csv(HF_FILE, low_memory=False)
df_haiku = pd.read_csv(HAIKU_FILE, low_memory=False)
df_gemini = pd.read_csv(GEMINI_FILE, low_memory=False)

for name, frame in [
    ("Hugging Face", df_hf),
    ("Haiku", df_haiku),
    ("Gemini", df_gemini)
]:
    if "record_key" not in frame.columns:
        raise ValueError(f"{name}: missing record_key")

    frame["record_key"] = frame["record_key"].astype(str)

    if not frame["record_key"].is_unique:
        raise ValueError(f"{name}: duplicate record_key")

# Restrict to reviews processed by both LLMs
common_ids = (
    set(df_haiku["record_key"])
    & set(df_gemini["record_key"])
    & set(df_hf["record_key"])
)

if len(common_ids) < 100:
    raise ValueError(
        f"Only {len(common_ids)} common reviews found. "
        "Both LLMs must process at least 100 identical reviews."
    )

# Prefer all common IDs when exactly 100.
# Otherwise select 100 reproducibly.
df_reviews = df_hf[
    df_hf["record_key"].isin(common_ids)
].copy()

# Apply the same exclusions to the validation set
for col in ["long_haul", "first_class"]:
    df_reviews[col] = pd.to_numeric(
        df_reviews[col], errors="coerce"
    )

df_reviews = df_reviews[
    (df_reviews["long_haul"] == 0)
    & (df_reviews["first_class"] == 0)
    & (df_reviews["review_en"].fillna("").str.strip() != "")
].copy()

if len(df_reviews) < 100:
    raise ValueError(
        f"Only {len(df_reviews)} eligible reviews remain."
    )

df_reviews = (
    df_reviews.sample(n=100, random_state=42)
    .sort_values("record_key")
    .reset_index(drop=True)
)

# Align LLM predictions
haiku_lookup = df_haiku.set_index("record_key")
gemini_lookup = df_gemini.set_index("record_key")

df_haiku_100 = haiku_lookup.loc[
    df_reviews["record_key"]
].reset_index()

df_gemini_100 = gemini_lookup.loc[
    df_reviews["record_key"]
].reset_index()

# Validate all required prediction columns before export
for feature in FEATURES:
    hf_col = f"{feature}_sentiment"
    llm_class_col = f"{feature}_llm"
    llm_sent_col = f"{feature}_sentiment_llm"

    if feature not in df_reviews or hf_col not in df_reviews:
        raise ValueError(f"Missing Hugging Face data for {feature}")

    for model_name, frame in [
        ("Haiku", df_haiku_100),
        ("Gemini", df_gemini_100)
    ]:
        for col in [llm_class_col, llm_sent_col]:
            if col not in frame:
                raise ValueError(
                    f"{model_name}: missing column {col}"
                )

for model_name, frame in [
    ("Hugging Face", df_reviews),
    ("Haiku", df_haiku_100),
    ("Gemini", df_gemini_100)
]:
    if "overall_sentiment" not in frame:
        raise ValueError(
            f"{model_name}: missing overall_sentiment"
        )

print("Common eligible reviews selected:", len(df_reviews))
print("Review IDs aligned:", (
    df_reviews["record_key"].tolist()
    == df_haiku_100["record_key"].tolist()
    == df_gemini_100["record_key"].tolist()
))

In [ ]:
wb = Workbook()

ws_overall = wb.active
ws_overall.title = "Overall Validation"

ws_aspect = wb.create_sheet("Aspect Validation")
ws_predictions = wb.create_sheet("Model Predictions")
ws_guide = wb.create_sheet("Instructions", 0)

# ---------------------------------------------
# INSTRUCTIONS
# ---------------------------------------------

instructions = [
    ["MODEL VALIDATION — AEGEAN PROJECT"],
    [""],
    ["1. Complete Overall Validation first."],
    ["2. Read each review and assign overall sentiment."],
    ["3. Complete Aspect Validation independently."],
    ["4. Use 1 when the feature is supported, 0 otherwise."],
    ["5. Use U when the evidence is genuinely ambiguous."],
    ["6. For absent aspects, use not_mentioned."],
    ["7. For present experience aspects, use neutral if discussed without clear polarity."],
    ["8. Mixed requires both positive and negative evidence."],
    ["9. Do not inspect Model Predictions until manual labeling is finished."],
    ["10. Do not use the original star rating to infer text sentiment."],
    [""],
    ["For contextual features, not_mentioned may be appropriate even when present."],
    ["A zero means insufficient positive evidence, not proof of the opposite."],
    ["Use the metadata where necessary to establish flight class or route."],
]

for row in instructions:
    ws_guide.append(row)

ws_guide.column_dimensions["A"].width = 110

# ---------------------------------------------
# OVERALL VALIDATION
# ---------------------------------------------

overall_headers = [
    "record_key",
    "airline",
    "source",
    "review_en",
    "manual_overall_sentiment",
    "validation_notes"
]

ws_overall.append(overall_headers)

for _, row in df_reviews.iterrows():
    ws_overall.append([
        row["record_key"],
        row["airline"],
        row.get("source", ""),
        row["review_en"],
        None,
        None
    ])

# ---------------------------------------------
# ASPECT VALIDATION
# ---------------------------------------------

aspect_headers = [
    "record_key",
    "airline",
    "feature",
    "feature_definition",
    "review_en",
    "manual_classification",
    "manual_sentiment",
    "validation_notes"
]

ws_aspect.append(aspect_headers)

for _, row in df_reviews.iterrows():
    for feature in FEATURES:
        ws_aspect.append([
            row["record_key"],
            row["airline"],
            feature,
            FEATURE_DESCRIPTIONS[feature],
            row["review_en"],
            None,
            None,
            None
        ])

# ---------------------------------------------
# MODEL PREDICTIONS
# ---------------------------------------------

prediction_headers = [
    "record_key",
    "feature",
    "hf_classification",
    "hf_sentiment",
    "haiku_classification",
    "haiku_sentiment",
    "gemini_classification",
    "gemini_sentiment"
]

ws_predictions.append(prediction_headers)

for i, row in df_reviews.iterrows():

    h = df_haiku_100.iloc[i]
    g = df_gemini_100.iloc[i]

    for feature in FEATURES:
        ws_predictions.append([
            row["record_key"],
            feature,
            int(row[feature]),
            row[f"{feature}_sentiment"],
            int(h[f"{feature}_llm"]),
            h[f"{feature}_sentiment_llm"],
            int(g[f"{feature}_llm"]),
            g[f"{feature}_sentiment_llm"]
        ])

# Add overall predictions to a separate section
ws_predictions.append([])
ws_predictions.append([
    "record_key",
    "overall_hf",
    "overall_haiku",
    "overall_gemini"
])

for i, row in df_reviews.iterrows():
    ws_predictions.append([
        row["record_key"],
        row["overall_sentiment"],
        df_haiku_100.iloc[i]["overall_sentiment"],
        df_gemini_100.iloc[i]["overall_sentiment"]
    ])

# ---------------------------------------------
# EXCEL DROPDOWNS
# ---------------------------------------------

overall_dv = DataValidation(
    type="list",
    formula1='"positive,negative,neutral,mixed,U"',
    allow_blank=True
)

classification_dv = DataValidation(
    type="list",
    formula1='"0,1,U"',
    allow_blank=True
)

aspect_sentiment_dv = DataValidation(
    type="list",
    formula1='"positive,negative,neutral,mixed,not_mentioned,U"',
    allow_blank=True
)

ws_overall.add_data_validation(overall_dv)
overall_dv.add(
    f"E2:E{ws_overall.max_row}"
)

ws_aspect.add_data_validation(classification_dv)
classification_dv.add(
    f"F2:F{ws_aspect.max_row}"
)

ws_aspect.add_data_validation(aspect_sentiment_dv)
aspect_sentiment_dv.add(
    f"G2:G{ws_aspect.max_row}"
)

# ---------------------------------------------
# FORMATTING
# ---------------------------------------------

for ws in [ws_overall, ws_aspect, ws_predictions]:

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )
        cell.fill = PatternFill(
            "solid",
            fgColor="17365D"
        )

    ws.row_dimensions[1].height = 30

    for col in range(1, ws.max_column + 1):
        ws.column_dimensions[
            get_column_letter(col)
        ].width = 22

    for col_name in ["D", "E"]:
        if ws == ws_overall and col_name == "D":
            ws.column_dimensions[col_name].width = 90

        if ws == ws_aspect and col_name == "E":
            ws.column_dimensions[col_name].width = 90

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

# Highlight manual entry columns
for ws, columns in [
    (ws_overall, ["E", "F"]),
    (ws_aspect, ["F", "G", "H"])
]:
    for col in columns:
        for row_num in range(2, ws.max_row + 1):
            ws[f"{col}{row_num}"].fill = PatternFill(
                "solid",
                fgColor="FFF2CC"
            )

# Hide prediction sheet for blind annotation.
# It can be unhidden after labeling.
ws_predictions.sheet_state = "hidden"

wb.save(VALIDATION_FILE)

print("Workbook created:", VALIDATION_FILE)
print("Overall reviews:", ws_overall.max_row - 1)
print("Aspect labeling decisions:", ws_aspect.max_row - 1)

# Gemini model only

In [ ]:
import os
import json
import time
import random
import hashlib
import asyncio
import re

from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from collections import deque

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from google import genai
from google.genai import types


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_DIR = HF_DIR

INPUT_FILE = DATA_DIR / "df_final_absa.csv"

OUTPUT_DIR = LLM_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gemini-3.1-flash-lite"

# All eligible reviews
MAX_REVIEWS = None

# Parallel processing
CONCURRENT_REQUESTS = 30

# Initial request-rate ceiling.
# This is NOT a claim about your actual Gemini quota.
REQUESTS_PER_MINUTE = 600

MAX_RETRIES = 6
BASE_RETRY_DELAY = 3
MAX_RETRY_DELAY = 120

MAX_OUTPUT_TOKENS = 4096
REQUEST_TIMEOUT_SECONDS = 120

# Display statistics every 100 completed reviews
PROGRESS_EVERY = 100

# USD per million tokens.
# Verify these rates against your provider's current pricing.
PRICE_INPUT = 0.25
PRICE_OUTPUT = 1.50

RANDOM_STATE = 42

CHECKPOINT_FILE = OUTPUT_DIR / "gemini_full_checkpoint.jsonl"
FAILURES_FILE = OUTPUT_DIR / "gemini_full_failures.jsonl"

OUTPUT_FILE = OUTPUT_DIR / "df_final_absa_gemini.csv"
DIAGNOSTICS_FILE = OUTPUT_DIR / "gemini_diagnostics.csv"
COST_FILE = OUTPUT_DIR / "gemini_cost_summary.csv"
COMPARISON_FILE = OUTPUT_DIR / "gemini_vs_huggingface.csv"

# ============================================================
# 2. FEATURES
# ============================================================

FEATURE_DEFINITIONS = {
    "long_haul":
        "Actual individual flight duration strictly greater "
        "than 6 hours. Do not use total journey duration.",

    "first_class":
        "Passenger actually traveled in first class.",

    "cancellation":
        "Actual flight cancellation discussed.",

    "delay":
        "Flight departure or arrival delay discussed.",

    "boarding":
        "Boarding process or boarding experience.",

    "baggage":
        "Luggage, baggage handling, allowance, or lost bags.",

    "economy_class":
        "Passenger actually traveled in economy class.",

    "business_class":
        "Passenger actually traveled in business class.",

    "international":
        "Flight crossing a national border.",

    "domestic":
        "Flight within one country.",

    "customer_service":
        "Customer support, telephone service, customer care.",

    "cabin_crew":
        "Flight attendants and onboard cabin crew.",

    "ground_staff":
        "Airline employees working at the airport.",

    "complaint_resolution":
        "Complaints, compensation, refunds, claims, disputes.",

    "seat_comfort":
        "Seat comfort, legroom, recline, seat pitch.",

    "food_beverage":
        "Food, drinks, meals, snacks, onboard catering.",

    "cabin_cleanliness":
        "Cabin cleanliness, toilets, hygiene, sanitation.",

    "entertainment_wifi":
        "Inflight entertainment, Wi-Fi, connectivity.",

    "booking_experience":
        "Booking process, reservations, ticket changes.",

    "check_in":
        "Online or airport check-in experience.",

    "airport_experience":
        "Airport lounges, terminals, facilities, experience.",

    "value_for_money":
        "Whether the experience was worth the money paid.",

    "pricing_fees":
        "Ticket pricing, charges, fees, ancillary costs.",

    "flight_disruption":
        "Missed connections, diversions, operational "
        "disruption, involuntary rebooking.",

    "overall_experience":
        "Explicit overall assessment of airline or journey."
}

FEATURES = list(FEATURE_DEFINITIONS)

SENTIMENT_LABELS = {
    "positive", "negative", "neutral",
    "mixed", "not_mentioned"
}

OVERALL_LABELS = {
    "positive", "negative", "neutral", "mixed"
}

FEATURE_GUIDE = "\n".join(
    f"{i}. {feature}: {definition}"
    for i, (feature, definition)
    in enumerate(FEATURE_DEFINITIONS.items(), start=1)
)

# ============================================================
# 3. CLASSIFICATION PROMPT
# ============================================================

# Keep this prompt unchanged between runs if you want
# existing checkpoints to remain valid.

SYSTEM_PROMPT = f"""
You are an expert annotator of airline passenger reviews.

Analyze the review independently, without access to predictions
from other classifiers.

You must produce:
- 25 binary feature classifications.
- Sentiment for each of the 25 features.
- One overall review sentiment.

FEATURES:
{FEATURE_GUIDE}

BINARY CLASSIFICATION:

1 = sufficient evidence that the feature applies.
0 = insufficient evidence.

A zero does not prove the opposite.

Only classify first_class, economy_class, and business_class
when the passenger actually traveled in that class.
Merely mentioning another cabin does not establish travel in it.

Long-haul means an individual flight lasting strictly
more than six hours. Never infer long-haul solely from
total connecting journey duration.

Use supplied route or flight metadata when it provides
reliable evidence. Do not invent routes or flight durations.

ASPECT SENTIMENT:

Allowed labels:
positive, negative, neutral, mixed, not_mentioned.

positive = favorable assessment of that aspect.
negative = unfavorable assessment.
neutral = discussed without clear favorable or unfavorable opinion.
mixed = both positive and negative evidence for that aspect.
not_mentioned = no relevant aspect sentiment or discussion.

For experience aspects, a factual mention without polarity
is neutral.

For contextual flight characteristics (cabin class,
international, domestic, long-haul), not_mentioned may
be appropriate even if classification is 1.

If classification is 0, sentiment MUST be not_mentioned.

Do not transfer the sentiment of one aspect to another.
For example, praise of cabin crew does not imply praise
of customer service.

OVERALL SENTIMENT:

Allowed labels:
positive, negative, neutral, mixed.

Evaluate the passenger's overall expressed experience,
considering the complete review.

Do not infer overall sentiment from airline name or
numerical star ratings.

OUTPUT:

Return valid JSON only.

The object must contain:
- overall_sentiment
- features

Every feature must contain:
- classification (integer 0 or 1)
- sentiment (one of the five allowed labels)

Include exactly all 25 named features.
Do not include commentary or markdown.
"""

PROMPT_VERSION = hashlib.sha256(
    SYSTEM_PROMPT.encode("utf-8")
).hexdigest()[:16]

# ============================================================
# 4. API CLIENT
# ============================================================

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    api_key = getpass("Enter Gemini API key: ")

client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        retry_options=types.HttpRetryOptions(
            attempts=1
        )
    )
)

print("Model:", MODEL_NAME)
print("Concurrency:", CONCURRENT_REQUESTS)
print("RPM ceiling:", REQUESTS_PER_MINUTE)
print("Prompt version:", PROMPT_VERSION)

In [ ]:
# ============================================================
# 5. DATA LOADING
# ============================================================

df_source = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

required_columns = {
    "record_key",
    "airline",
    "review_en",
    "long_haul",
    "first_class"
}

missing = required_columns - set(df_source.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

df_source["record_key"] = (
    df_source["record_key"].astype("string")
)

if df_source["record_key"].isna().any():
    raise ValueError("Missing record_key values")

if not df_source["record_key"].is_unique:
    raise ValueError("Duplicate record_key values")

df_source["record_key"] = (
    df_source["record_key"].astype(str)
)

df_source["review_en"] = (
    df_source["review_en"]
    .fillna("")
    .astype(str)
)

for feature in ["long_haul", "first_class"]:

    df_source[feature] = pd.to_numeric(
        df_source[feature],
        errors="coerce"
    )

    if not df_source[feature].dropna().isin([0, 1]).all():
        raise ValueError(
            f"Invalid values in {feature}"
        )

df_work = df_source.loc[
    (df_source["long_haul"].fillna(0) != 1)
    & (df_source["first_class"].fillna(0) != 1)
    & (df_source["review_en"].str.strip() != "")
    & pd.to_datetime(df_source["review_date"], errors="coerce", utc=True).ge(pd.Timestamp("2024-01-01", tz="UTC"))
].copy()

df_work = (
    df_work.sort_values("record_key")
    .reset_index(drop=True)
)

if MAX_REVIEWS is not None:

    df_work = (
        df_work.sample(
            n=min(MAX_REVIEWS, len(df_work)),
            random_state=RANDOM_STATE
        )
        .sort_values("record_key")
        .reset_index(drop=True)
    )

# ============================================================
# 6. PAYLOAD AND VALIDATION
# ============================================================

METADATA_COLUMNS = [
    "origin",
    "destination",
    "route",
    "flight_number",
    "travel_class"
]


def build_payload(row):

    """Return or process build payload."""
    metadata = {}

    for column in METADATA_COLUMNS:

        if column not in row.index:
            continue

        value = row[column]

        if pd.notna(value):
            value = str(value).strip()

            if value:
                metadata[column] = value

    return json.dumps(
        {
            "review": row["review_en"],
            "metadata": metadata
        },
        ensure_ascii=False
    )


def hash_text(text):

    """Return or process hash text."""
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def validate_prediction(result):

    """Return or process validate prediction."""
    if not isinstance(result, dict):
        raise ValueError("Invalid JSON object")

    if result.get("overall_sentiment") not in OVERALL_LABELS:
        raise ValueError("Invalid overall sentiment")

    predictions = result.get("features")

    if not isinstance(predictions, dict):
        raise ValueError("Missing features object")

    if set(predictions) != set(FEATURES):
        raise ValueError(
            "Feature mismatch: "
            f"{set(predictions) ^ set(FEATURES)}"
        )

    for feature in FEATURES:

        item = predictions[feature]

        if not isinstance(item, dict):
            raise ValueError(
                f"{feature}: invalid object"
            )

        binary = item.get("classification")
        sentiment = item.get("sentiment")

        if type(binary) is not int or binary not in (0, 1):
            raise ValueError(
                f"{feature}: invalid classification"
            )

        if sentiment not in SENTIMENT_LABELS:
            raise ValueError(
                f"{feature}: invalid sentiment"
            )

        if binary == 0 and sentiment != "not_mentioned":
            raise ValueError(
                f"{feature}: inconsistent classification"
            )

    return result


# ============================================================
# 7. CHECKPOINT FUNCTIONS
# ============================================================

def load_checkpoint():

    """Return or process load checkpoint."""
    records = {}

    if not CHECKPOINT_FILE.exists():
        return records

    with CHECKPOINT_FILE.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if not line.strip():
                continue

            try:
                item = json.loads(line)

                if item.get("status") == "success":
                    records[str(item["record_key"])] = item

            except (ValueError, KeyError):
                continue

    return records


def is_current(item, payload_hash):

    """Return or process is current."""
    return (
        item is not None
        and item.get("model_id") == MODEL_NAME
        and item.get("prompt_version") == PROMPT_VERSION
        and item.get("payload_hash") == payload_hash
        and item.get("status") == "success"
    )


completed = load_checkpoint()

reusable = 0

for _, row in df_work.iterrows():

    key = str(row["record_key"])

    payload_hash = hash_text(
        build_payload(row)
    )

    if is_current(completed.get(key), payload_hash):
        reusable += 1

print("=" * 60)
print("DATASET AND CHECKPOINT STATUS")
print("=" * 60)

print("Original reviews:", f"{len(df_source):,}")
print("Eligible reviews:", f"{len(df_work):,}")
print("Checkpoint entries:", f"{len(completed):,}")
print("Reusable completed reviews:", f"{reusable:,}")
print("Remaining:", f"{len(df_work) - reusable:,}")

In [ ]:
# ============================================================
# 8. ASYNC RATE LIMITER
# ============================================================

class AsyncRateLimiter:

    """Implement AsyncRateLimiter."""
    def __init__(self, rpm):

        """Return or process init."""
        if rpm <= 0:
            raise ValueError("RPM must be positive")

        self.rpm = rpm
        self.interval = 60.0 / rpm
        self.next_allowed = 0.0
        self.lock = asyncio.Lock()

    async def wait(self):

        """Return or process wait."""
        async with self.lock:

            now = time.monotonic()

            scheduled = max(
                now,
                self.next_allowed
            )

            self.next_allowed = (
                scheduled + self.interval
            )

        delay = scheduled - now

        if delay > 0:
            await asyncio.sleep(delay)


# ============================================================
# 9. ASYNC GEMINI REQUEST
# ============================================================

async def call_gemini_async(payload):

    """Return or process call gemini async."""
    response = await client.aio.models.generate_content(
        model=MODEL_NAME,
        contents=payload,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            max_output_tokens=MAX_OUTPUT_TOKENS,
            temperature=0,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    if not response.text:
        raise ValueError("Empty Gemini response")

    result = validate_prediction(
        json.loads(response.text)
    )

    usage = response.usage_metadata

    if usage is None:
        raise ValueError("Missing usage metadata")

    input_tokens = int(
        usage.prompt_token_count or 0
    )

    output_tokens = int(
        usage.candidates_token_count or 0
    )

    thinking_tokens = int(
        usage.thoughts_token_count or 0
    )

    billed_output_tokens = (
        output_tokens + thinking_tokens
    )

    cost = (
        input_tokens * PRICE_INPUT
        + billed_output_tokens * PRICE_OUTPUT
    ) / 1_000_000

    return {
        "result": result,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "thinking_tokens": thinking_tokens,
        "billed_output_tokens": billed_output_tokens,
        "estimated_cost_usd": cost
    }


# ============================================================
# 10. RETRY HANDLING
# ============================================================

def get_status_code(exc):

    """Return or process get status code."""
    for attr in ["code", "status_code"]:

        value = getattr(exc, attr, None)

        try:
            if value is not None:
                return int(value)
        except (TypeError, ValueError):
            pass

    return None


def retryable_error(exc):

    """Return or process retryable error."""
    code = get_status_code(exc)

    if code in (400, 401, 403, 404):
        return False

    if code is not None:
        return code in (408, 409, 429) or code >= 500

    # Retry transient network errors and invalid responses.
    return isinstance(
        exc,
        (
            TimeoutError,
            ConnectionError,
            OSError,
            ValueError,
            json.JSONDecodeError
        )
    )


async def call_with_retries_async(
    payload,
    rate_limiter
):

    """Return or process call with retries async."""
    for attempt in range(MAX_RETRIES):

        try:

            await rate_limiter.wait()

            return await call_gemini_async(
                payload
            )

        except Exception as exc:

            if (
                attempt == MAX_RETRIES - 1
                or not retryable_error(exc)
            ):
                raise

            delay = min(
                MAX_RETRY_DELAY,
                BASE_RETRY_DELAY * (2 ** attempt)
                + random.uniform(0, 3)
            )

            await asyncio.sleep(delay)

In [ ]:
# ============================================================
# 11. FULL ASYNCHRONOUS PROCESSING
# ============================================================

async def run_full_gemini_async(dataframe):

    """Return or process run full gemini async."""
    completed = load_checkpoint()

    pending = []
    skipped = 0

    # Identify remaining reviews.
    for _, row in dataframe.iterrows():

        record_key = str(row["record_key"])
        payload = build_payload(row)
        payload_hash = hash_text(payload)

        if is_current(
            completed.get(record_key),
            payload_hash
        ):
            skipped += 1
        else:
            pending.append((
                record_key,
                payload,
                payload_hash
            ))

    total_pending = len(pending)

    print("\n" + "=" * 65)
    print("GEMINI 3.1 FLASH-LITE — PARALLEL ANALYSIS")
    print("=" * 65)

    print("Eligible reviews:", f"{len(dataframe):,}")
    print("Already completed:", f"{skipped:,}")
    print("Remaining:", f"{total_pending:,}")
    print("Concurrent requests:", CONCURRENT_REQUESTS)
    print("RPM ceiling:", REQUESTS_PER_MINUTE)

    if total_pending == 0:

        print("All reviews already completed.")

        return {
            "successes": 0,
            "failures": 0,
            "skipped": skipped,
            "session_cost": 0.0
        }

    # A bounded queue limits memory use.
    work_queue = asyncio.Queue(
        maxsize=CONCURRENT_REQUESTS * 3
    )

    result_queue = asyncio.Queue()

    rate_limiter = AsyncRateLimiter(
        REQUESTS_PER_MINUTE
    )

    successes = 0
    failures = 0
    session_cost = 0.0

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # WORKER
    # --------------------------------------------------------

    async def worker(worker_id):

        """Return or process worker."""
        while True:

            item = await work_queue.get()

            try:

                if item is None:
                    return

                record_key, payload, payload_hash = item

                request_start = time.perf_counter()

                try:

                    response = await call_with_retries_async(
                        payload,
                        rate_limiter
                    )

                    elapsed = (
                        time.perf_counter()
                        - request_start
                    )

                    record = {
                        "record_key": record_key,
                        "model_id": MODEL_NAME,
                        "prompt_version": PROMPT_VERSION,
                        "payload_hash": payload_hash,
                        "status": "success",
                        "timestamp_utc": (
                            datetime.now(timezone.utc)
                            .isoformat()
                        ),
                        "result": response["result"],
                        "input_tokens": response["input_tokens"],
                        "output_tokens": response["output_tokens"],
                        "thinking_tokens": response["thinking_tokens"],
                        "billed_output_tokens": response[
                            "billed_output_tokens"
                        ],
                        "estimated_cost_usd": response[
                            "estimated_cost_usd"
                        ],
                        "elapsed_seconds": elapsed
                    }

                    await result_queue.put(
                        ("success", record)
                    )

                except Exception as exc:

                    failure = {
                        "record_key": record_key,
                        "timestamp_utc": (
                            datetime.now(timezone.utc)
                            .isoformat()
                        ),
                        "error_type": type(exc).__name__,
                        "error": str(exc),
                        "status_code": get_status_code(exc)
                    }

                    await result_queue.put(
                        ("failure", failure)
                    )

            finally:
                work_queue.task_done()

    # --------------------------------------------------------
    # SINGLE CHECKPOINT WRITER
    # --------------------------------------------------------

    async def writer():

        """Return or process writer."""
        nonlocal successes, failures, session_cost

        with (
            CHECKPOINT_FILE.open(
                "a",
                encoding="utf-8"
            ) as checkpoint,
            FAILURES_FILE.open(
                "a",
                encoding="utf-8"
            ) as failure_log
        ):

            with tqdm(
                total=total_pending,
                desc="Gemini 30 concurrent",
                unit="review"
            ) as progress:

                while True:

                    item = await result_queue.get()

                    try:

                        if item is None:
                            return

                        status, record = item

                        if status == "success":

                            checkpoint.write(
                                json.dumps(
                                    record,
                                    ensure_ascii=False
                                ) + "\n"
                            )

                            checkpoint.flush()

                            completed[
                                record["record_key"]
                            ] = record

                            successes += 1

                            session_cost += (
                                record["estimated_cost_usd"]
                            )

                        else:

                            failure_log.write(
                                json.dumps(
                                    record,
                                    ensure_ascii=False
                                ) + "\n"
                            )

                            failure_log.flush()

                            failures += 1

                        progress.update(1)

                        elapsed = max(
                            time.perf_counter() - start_time,
                            0.001
                        )

                        processed = successes + failures

                        throughput = (
                            processed / elapsed * 60
                        )

                        remaining = (
                            total_pending - processed
                        )

                        eta_minutes = (
                            remaining / throughput
                            if throughput > 0
                            else float("inf")
                        )

                        progress.set_postfix({
                            "RPM": f"{throughput:.1f}",
                            "failed": failures,
                            "cost": f"${session_cost:.3f}",
                            "ETA_min": f"{eta_minutes:.1f}"
                        })

                        if (
                            processed % PROGRESS_EVERY == 0
                        ):

                            print(
                                f"\nProcessed: {processed:,}"
                                f"/{total_pending:,}"
                                f" | Successes: {successes:,}"
                                f" | Failures: {failures:,}"
                                f" | RPM: {throughput:.1f}"
                                f" | Cost: ${session_cost:.4f}"
                            )

                    finally:
                        result_queue.task_done()

    # --------------------------------------------------------
    # PRODUCER
    # --------------------------------------------------------

    async def producer():

        """Return or process producer."""
        for item in pending:
            await work_queue.put(item)

        for _ in range(CONCURRENT_REQUESTS):
            await work_queue.put(None)

    # --------------------------------------------------------
    # EXECUTION
    # --------------------------------------------------------

    workers = [
        asyncio.create_task(worker(i))
        for i in range(CONCURRENT_REQUESTS)
    ]

    writer_task = asyncio.create_task(writer())
    producer_task = asyncio.create_task(producer())

    try:

        await producer_task

        await work_queue.join()

        await asyncio.gather(*workers)

        await result_queue.put(None)
        await result_queue.join()

        await writer_task

    except BaseException:

        # Cancel outstanding work.
        producer_task.cancel()

        for task in workers:
            task.cancel()

        await asyncio.gather(
            producer_task,
            *workers,
            return_exceptions=True
        )

        # Allow the writer to save any completed results
        # that were already placed on the result queue.
        if not writer_task.done():

            await result_queue.put(None)
            await result_queue.join()
            await writer_task

        raise

    elapsed = time.perf_counter() - start_time

    print("\n" + "=" * 65)
    print("RUN FINISHED")
    print("=" * 65)

    print("New successes:", f"{successes:,}")
    print("Failures:", f"{failures:,}")
    print("Previously completed:", f"{skipped:,}")
    print("Duration:", f"{elapsed / 60:.2f} minutes")

    if elapsed > 0:
        print(
            "Successful throughput:",
            f"{successes / elapsed * 60:.1f} reviews/min"
        )

    print(
        "Estimated session cost:",
        f"${session_cost:.4f}"
    )

    return {
        "successes": successes,
        "failures": failures,
        "skipped": skipped,
        "session_cost": session_cost,
        "elapsed_seconds": elapsed
    }

In [ ]:
# ============================================================
# 12. EXPORT RESULTS
# ============================================================

def export_gemini_results(dataframe):

    """Return or process export gemini results."""
    checkpoint = load_checkpoint()

    rows = []

    for _, row in dataframe.iterrows():

        key = str(row["record_key"])
        payload_hash = hash_text(build_payload(row))

        item = checkpoint.get(key)

        if not is_current(item, payload_hash):
            continue

        result = item["result"]

        flat = {
            "record_key": key,
            "gemini_model": item["model_id"],
            "gemini_overall_sentiment":
                result["overall_sentiment"],
            "gemini_input_tokens":
                item["input_tokens"],
            "gemini_output_tokens":
                item["output_tokens"],
            "gemini_thinking_tokens":
                item["thinking_tokens"],
            "gemini_billed_output_tokens":
                item["billed_output_tokens"],
            "gemini_cost_usd":
                item["estimated_cost_usd"],
            "gemini_elapsed_seconds":
                item["elapsed_seconds"]
        }

        for feature in FEATURES:

            prediction = result["features"][feature]

            flat[f"{feature}_llm"] = (
                prediction["classification"]
            )

            flat[f"{feature}_sentiment_llm"] = (
                prediction["sentiment"]
            )

        rows.append(flat)

    if not rows:
        print("No successful Gemini results yet.")
        return pd.DataFrame()

    df_predictions = pd.DataFrame(rows)

    df_output = dataframe.merge(
        df_predictions,
        on="record_key",
        how="inner",
        validate="one_to_one"
    )

    df_output.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # COST SUMMARY
    # --------------------------------------------------------

    cost_summary = pd.DataFrame([{
        "model": MODEL_NAME,
        "reviews_processed": len(df_output),
        "input_tokens":
            df_output["gemini_input_tokens"].sum(),
        "output_tokens":
            df_output["gemini_output_tokens"].sum(),
        "thinking_tokens":
            df_output["gemini_thinking_tokens"].sum(),
        "billed_output_tokens":
            df_output["gemini_billed_output_tokens"].sum(),
        "estimated_cost_usd":
            df_output["gemini_cost_usd"].sum(),
        "average_cost_per_review_usd":
            df_output["gemini_cost_usd"].mean(),
        "average_seconds_per_review":
            df_output["gemini_elapsed_seconds"].mean()
    }])

    cost_summary.to_csv(
        COST_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # FEATURE COMPARISON
    # --------------------------------------------------------

    diagnostics = []

    for feature in FEATURES:

        hf_sentiment_col = f"{feature}_sentiment"

        if (
            feature not in df_output.columns
            or hf_sentiment_col not in df_output.columns
        ):
            continue

        temp = pd.DataFrame({
            "record_key": df_output["record_key"],
            "airline": df_output["airline"],
            "feature": feature,
            "hf_classification": df_output[feature],
            "gemini_classification":
                df_output[f"{feature}_llm"],
            "hf_sentiment":
                df_output[hf_sentiment_col],
            "gemini_sentiment":
                df_output[f"{feature}_sentiment_llm"]
        })

        temp["binary_agreement"] = (
            temp["hf_classification"]
            == temp["gemini_classification"]
        )

        temp["sentiment_agreement"] = (
            temp["hf_sentiment"]
            == temp["gemini_sentiment"]
        )

        diagnostics.append(temp)

    if diagnostics:

        df_diagnostics = pd.concat(
            diagnostics,
            ignore_index=True
        )

        df_diagnostics.to_csv(
            DIAGNOSTICS_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        df_comparison = (
            df_diagnostics.groupby("feature")
            .agg(
                binary_agreement=("binary_agreement", "mean"),
                sentiment_agreement=("sentiment_agreement", "mean"),
                observations=("record_key", "size")
            )
            .reset_index()
        )

        df_comparison.to_csv(
            COMPARISON_FILE,
            index=False,
            encoding="utf-8-sig"
        )

    print("\nExport completed.")
    print("Reviews exported:", f"{len(df_output):,}")
    print("Main file:", OUTPUT_FILE)
    print("Cost file:", COST_FILE)
    print(
        "Estimated cost of exported successful requests:",
        f"${df_output['gemini_cost_usd'].sum():.4f}"
    )

    return df_output

In [ ]:
# ============================================================
# 13. RUN FULL GEMINI ANALYSIS
# ============================================================

print("Starting 30 concurrent Gemini requests...")

try:

    run_statistics = await run_full_gemini_async(
        df_work
    )

finally:

    # Also export after interruption or failure.
    # The JSONL checkpoint is the authoritative
    # source of successfully completed requests.
    df_gemini = export_gemini_results(
        df_work
    )

print("\nDataset shape:", df_gemini.shape)

In [ ]:
CONCURRENT_REQUESTS = 10
REQUESTS_PER_MINUTE = 100

run_statistics = await run_full_gemini_async(df_work)

df_gemini = export_gemini_results(df_work)

In [ ]:
import json
import pandas as pd

from collections import Counter

# ------------------------------------------------------------
# LOAD FAILURE LOG
# ------------------------------------------------------------

failures = []

with open(
    FAILURES_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():
            failures.append(json.loads(line))

df_failures = pd.DataFrame(failures)

print("Total failure attempts:", len(df_failures))
print("Unique failed reviews:", df_failures["record_key"].nunique())

# Keep the latest failure for each review.
df_latest_failures = (
    df_failures
    .drop_duplicates(
        subset="record_key",
        keep="last"
    )
    .copy()
)

print("\nFAILURE TYPES")
display(
    df_latest_failures.groupby(
        ["error_type", "status_code"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nMOST COMMON ERROR MESSAGES")
display(
    df_latest_failures["error"]
    .value_counts()
    .head(20)
)

print("\nEXAMPLE FAILURES")
display(
    df_latest_failures[
        ["record_key", "error_type", "status_code", "error"]
    ].head(15)
)

In [ ]:
def retryable_error(exc):

    """Return or process retryable error."""
    code = get_status_code(exc)

    # Permanent errors: do not retry.
    if code in (400, 401, 402, 403, 404):
        return False

    # Temporary API errors.
    if code is not None:
        return code in (408, 409, 429) or code >= 500

    return isinstance(
        exc,
        (
            TimeoutError,
            ConnectionError,
            OSError,
            ValueError,
            json.JSONDecodeError
        )
    )

In [ ]:
CONCURRENT_REQUESTS = 10
REQUESTS_PER_MINUTE = 100

run_statistics = await run_full_gemini_async(df_work)

df_gemini = export_gemini_results(df_work)

print(
    f"Completed: {len(df_gemini):,} "
    f"/ {len(df_work):,}"
)

In [ ]:
print("Kernel is responding")

In [ ]:
completed = load_checkpoint()

print("Successful reviews in checkpoint:", len(completed))
print("Eligible reviews:", len(df_work))
print("Remaining reviews:", len(df_work) - len(completed))

In [ ]:
import pandas as pd

df_gemini = pd.read_csv(
    OUTPUT_FILE,
    low_memory=False
)

print("Gemini dataset shape:", df_gemini.shape)
print("Unique reviews:", df_gemini["record_key"].nunique())

In [ ]:
# ============================================================
# RECOVER ONLY THE 65 MISSING GEMINI REVIEWS
# ============================================================

import pandas as pd

# Load successful checkpoint records
completed = load_checkpoint()

# Identify reviews without a valid completed prediction
missing_indices = []

for idx, row in df_work.iterrows():

    record_key = str(row["record_key"])
    payload_hash = hash_text(build_payload(row))

    if not is_current(
        completed.get(record_key),
        payload_hash
    ):
        missing_indices.append(idx)

df_missing = df_work.loc[
    missing_indices
].copy().reset_index(drop=True)

print("=" * 60)
print("GEMINI RECOVERY")
print("=" * 60)

print("Total eligible reviews:", len(df_work))
print("Previously completed:", len(df_work) - len(df_missing))
print("Reviews remaining:", len(df_missing))

display(
    df_missing[
        ["record_key", "airline"]
    ].head(10)
)

In [ ]:
# ============================================================
# PROCESS ONLY THE MISSING REVIEWS
# ============================================================

import asyncio
import json
import time

from datetime import datetime, timezone
from tqdm.auto import tqdm

RECOVERY_CONCURRENCY = 5
RECOVERY_RPM = 60

async def recover_missing_reviews(df_missing):

    """Return or process recover missing reviews."""
    limiter = AsyncRateLimiter(
        RECOVERY_RPM
    )

    semaphore = asyncio.Semaphore(
        RECOVERY_CONCURRENCY
    )

    successes = 0
    failures = 0
    session_cost = 0.0

    async def process_review(row):

        """Return or process process review."""
        async with semaphore:

            record_key = str(row["record_key"])
            payload = build_payload(row)
            payload_hash = hash_text(payload)

            start = time.perf_counter()

            try:

                response = await call_with_retries_async(
                    payload,
                    limiter
                )

                elapsed = time.perf_counter() - start

                record = {
                    "record_key": record_key,
                    "model_id": MODEL_NAME,
                    "prompt_version": PROMPT_VERSION,
                    "payload_hash": payload_hash,
                    "status": "success",
                    "timestamp_utc": (
                        datetime.now(timezone.utc)
                        .isoformat()
                    ),
                    "result": response["result"],
                    "input_tokens": response["input_tokens"],
                    "output_tokens": response["output_tokens"],
                    "thinking_tokens": response["thinking_tokens"],
                    "billed_output_tokens": response[
                        "billed_output_tokens"
                    ],
                    "estimated_cost_usd": response[
                        "estimated_cost_usd"
                    ],
                    "elapsed_seconds": elapsed
                }

                return ("success", record)

            except Exception as exc:

                return (
                    "failure",
                    {
                        "record_key": record_key,
                        "error_type": type(exc).__name__,
                        "status_code": get_status_code(exc),
                        "error": str(exc)
                    }
                )

    tasks = [
        asyncio.create_task(process_review(row))
        for _, row in df_missing.iterrows()
    ]

    # Only the main coroutine writes to files.
    with (
        CHECKPOINT_FILE.open("a", encoding="utf-8") as checkpoint,
        FAILURES_FILE.open("a", encoding="utf-8") as failure_log
    ):

        try:

            for task in tqdm(
                asyncio.as_completed(tasks),
                total=len(tasks),
                desc="Recovering Gemini reviews"
            ):

                status, record = await task

                if status == "success":

                    checkpoint.write(
                        json.dumps(
                            record,
                            ensure_ascii=False
                        ) + "\n"
                    )
                    checkpoint.flush()

                    successes += 1
                    session_cost += record[
                        "estimated_cost_usd"
                    ]

                else:

                    failure_log.write(
                        json.dumps(
                            record,
                            ensure_ascii=False
                        ) + "\n"
                    )
                    failure_log.flush()

                    failures += 1

        finally:

            for task in tasks:
                if not task.done():
                    task.cancel()

            await asyncio.gather(
                *tasks,
                return_exceptions=True
            )

    print("\nRECOVERY COMPLETED")
    print("New successes:", successes)
    print("Failures:", failures)
    print("Estimated additional cost:", f"${session_cost:.4f}")

    return {
        "successes": successes,
        "failures": failures,
        "session_cost": session_cost
    }


# Execute only the 65 missing reviews
recovery_stats = await recover_missing_reviews(
    df_missing
)

In [ ]:
# ============================================================
# TEST GEMINI ON ONE MISSING REVIEW
# ============================================================

import time

test_row = df_missing.iloc[0]

print("Testing review:", test_row["record_key"])

payload = build_payload(test_row)

start = time.perf_counter()

try:
    response = call_gemini(payload)

    print("SUCCESS")
    print(
        "Response time:",
        round(time.perf_counter() - start, 2),
        "seconds"
    )

    print(
        "Overall sentiment:",
        response["result"]["overall_sentiment"]
    )

    print(
        "Estimated cost:",
        response["estimated_cost_usd"]
    )

except Exception as exc:

    print("ERROR:", type(exc).__name__)
    print(str(exc))

In [ ]:
def validate_prediction(result):
    """
    Validate and normalize Gemini ABSA predictions.

    Rules:
    - Exactly 25 features must be present.
    - Binary classifications must be 0 or 1.
    - Sentiment labels must be valid.
    - If classification = 0, sentiment is normalized
      to 'not_mentioned'.

    Returns the validated, normalized prediction.
    """

    if not isinstance(result, dict):
        raise ValueError("Invalid JSON object")

    overall = result.get("overall_sentiment")

    if overall not in OVERALL_LABELS:
        raise ValueError(
            f"Invalid overall sentiment: {overall}"
        )

    predictions = result.get("features")

    if not isinstance(predictions, dict):
        raise ValueError("Missing features object")

    if set(predictions) != set(FEATURES):
        raise ValueError(
            "Feature mismatch: "
            f"{set(predictions) ^ set(FEATURES)}"
        )

    normalized = {
        "overall_sentiment": overall,
        "features": {}
    }

    corrections = []

    for feature in FEATURES:

        item = predictions[feature]

        if not isinstance(item, dict):
            raise ValueError(
                f"{feature}: invalid object"
            )

        binary = item.get("classification")
        sentiment = item.get("sentiment")

        if type(binary) is not int or binary not in (0, 1):
            raise ValueError(
                f"{feature}: invalid classification"
            )

        if sentiment not in SENTIMENT_LABELS:
            raise ValueError(
                f"{feature}: invalid sentiment"
            )

        if binary == 0 and sentiment != "not_mentioned":

            corrections.append({
                "feature": feature,
                "original_sentiment": sentiment,
                "corrected_sentiment": "not_mentioned"
            })

            sentiment = "not_mentioned"

        normalized["features"][feature] = {
            "classification": binary,
            "sentiment": sentiment
        }

    normalized["validation_corrections"] = corrections

    return normalized

In [ ]:
test_row = df_missing.iloc[0]

payload = build_payload(test_row)

response = call_gemini(payload)

print("SUCCESS")

print(
    "Overall sentiment:",
    response["result"]["overall_sentiment"]
)

print(
    "Automatic corrections:",
    response["result"]["validation_corrections"]
)

print(
    "Estimated cost:",
    response["estimated_cost_usd"]
)

In [ ]:
# ============================================================
# GEMINI RECOVERY — REMAINING 65 REVIEWS
# ============================================================

import json
import time
from datetime import datetime, timezone
from tqdm.auto import tqdm


def recover_gemini_sequential(df_missing):

    """Return or process recover gemini sequential."""
    completed = load_checkpoint()

    successes = 0
    failures = 0
    skipped = 0
    total_cost = 0.0

    with (
        CHECKPOINT_FILE.open("a", encoding="utf-8") as checkpoint,
        FAILURES_FILE.open("a", encoding="utf-8") as failure_log
    ):

        for _, row in tqdm(
            df_missing.iterrows(),
            total=len(df_missing),
            desc="Recovering Gemini reviews"
        ):

            record_key = str(row["record_key"])

            payload = build_payload(row)
            payload_hash = hash_text(payload)

            # Skip reviews already successfully processed
            if is_current(
                completed.get(record_key),
                payload_hash
            ):
                skipped += 1
                continue

            start = time.perf_counter()

            try:

                # Uses the updated validate_prediction()
                response = call_gemini(payload)

                elapsed = time.perf_counter() - start

                record = {
                    "record_key": record_key,
                    "model_id": MODEL_NAME,
                    "prompt_version": PROMPT_VERSION,
                    "payload_hash": payload_hash,
                    "status": "success",
                    "timestamp_utc": (
                        datetime.now(timezone.utc).isoformat()
                    ),
                    "result": response["result"],
                    "input_tokens": response["input_tokens"],
                    "output_tokens": response["output_tokens"],
                    "thinking_tokens": response["thinking_tokens"],
                    "billed_output_tokens": response[
                        "billed_output_tokens"
                    ],
                    "estimated_cost_usd": response[
                        "estimated_cost_usd"
                    ],
                    "elapsed_seconds": elapsed
                }

                # Save immediately to existing checkpoint
                checkpoint.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                checkpoint.flush()

                completed[record_key] = record

                successes += 1

                total_cost += response[
                    "estimated_cost_usd"
                ]

            except Exception as exc:

                failures += 1

                failure_record = {
                    "record_key": record_key,
                    "timestamp_utc": (
                        datetime.now(timezone.utc).isoformat()
                    ),
                    "error_type": type(exc).__name__,
                    "status_code": get_status_code(exc),
                    "error": str(exc)
                }

                failure_log.write(
                    json.dumps(
                        failure_record,
                        ensure_ascii=False
                    ) + "\n"
                )

                failure_log.flush()

                print(
                    f"\nFAILED: {record_key[:16]}"
                    f" | {type(exc).__name__}"
                    f" | {str(exc)[:200]}"
                )

                # Stop immediately if credits run out
                if get_status_code(exc) == 402:
                    print("Prepaid credits exhausted. Stopping.")
                    break

            time.sleep(0.2)

    print("\n" + "=" * 60)
    print("GEMINI RECOVERY COMPLETED")
    print("=" * 60)

    print("New successes:", successes)
    print("Skipped:", skipped)
    print("Failures:", failures)
    print("Additional estimated cost:", f"${total_cost:.4f}")

    return {
        "successes": successes,
        "skipped": skipped,
        "failures": failures,
        "session_cost": total_cost
    }

In [ ]:
recovery_stats = recover_gemini_sequential(df_missing)

In [ ]:
from google.genai import types
import json

# ============================================================
# STRICT STRUCTURED OUTPUT SCHEMA
# ============================================================

FEATURE_SCHEMA = {
    "type": "object",
    "properties": {
        "classification": {
            "type": "integer",
            "enum": [0, 1]
        },
        "sentiment": {
            "type": "string",
            "enum": [
                "positive",
                "negative",
                "neutral",
                "mixed",
                "not_mentioned"
            ]
        }
    },
    "required": [
        "classification",
        "sentiment"
    ]
}

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "overall_sentiment": {
            "type": "string",
            "enum": [
                "positive",
                "negative",
                "neutral",
                "mixed"
            ]
        },
        "features": {
            "type": "object",
            "properties": {
                feature: FEATURE_SCHEMA
                for feature in FEATURES
            },
            "required": FEATURES
        }
    },
    "required": [
        "overall_sentiment",
        "features"
    ]
}


# ============================================================
# REPLACE ORIGINAL SYNCHRONOUS API FUNCTION
# ============================================================

def call_gemini(payload):

    """Return or process call gemini."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=payload,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            max_output_tokens=8192,
            temperature=0,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    if not response.text:
        raise ValueError(
            "Gemini returned an empty response"
        )

    result = validate_prediction(
        json.loads(response.text)
    )

    usage = response.usage_metadata

    if usage is None:
        raise ValueError("Missing usage metadata")

    input_tokens = int(
        usage.prompt_token_count or 0
    )

    output_tokens = int(
        usage.candidates_token_count or 0
    )

    thinking_tokens = int(
        usage.thoughts_token_count or 0
    )

    billed_output_tokens = (
        output_tokens + thinking_tokens
    )

    estimated_cost = (
        input_tokens * PRICE_INPUT
        + billed_output_tokens * PRICE_OUTPUT
    ) / 1_000_000

    return {
        "result": result,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "thinking_tokens": thinking_tokens,
        "billed_output_tokens": billed_output_tokens,
        "estimated_cost_usd": estimated_cost
    }

In [ ]:
# ============================================================
# IDENTIFY CURRENTLY MISSING REVIEWS
# ============================================================

completed = load_checkpoint()

missing_indices = []

for idx, row in df_work.iterrows():

    record_key = str(row["record_key"])
    payload_hash = hash_text(build_payload(row))

    if not is_current(
        completed.get(record_key),
        payload_hash
    ):
        missing_indices.append(idx)

df_missing = (
    df_work.loc[missing_indices]
    .copy()
    .reset_index(drop=True)
)

print("Remaining reviews:", len(df_missing))

display(
    df_missing[
        ["record_key", "airline"]
    ]
)

In [ ]:
recovery_stats = recover_gemini_sequential(
    df_missing
)

In [ ]:
# ============================================================
# FIX GEMINI STRUCTURED OUTPUT SCHEMA
# ============================================================

from google.genai import types

FEATURE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "classification": {
            "type": "INTEGER",
            "minimum": 0,
            "maximum": 1
        },
        "sentiment": {
            "type": "STRING",
            "enum": [
                "positive",
                "negative",
                "neutral",
                "mixed",
                "not_mentioned"
            ]
        }
    },
    "required": [
        "classification",
        "sentiment"
    ]
}

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "overall_sentiment": {
            "type": "STRING",
            "enum": [
                "positive",
                "negative",
                "neutral",
                "mixed"
            ]
        },
        "features": {
            "type": "OBJECT",
            "properties": {
                feature: FEATURE_SCHEMA
                for feature in FEATURES
            },
            "required": FEATURES
        }
    },
    "required": [
        "overall_sentiment",
        "features"
    ]
}

# Validate locally before making any API requests.
validated_schema = types.Schema.model_validate(
    RESPONSE_SCHEMA
)

print("Schema validation successful.")
print("Number of features:", len(FEATURES))

In [ ]:
# ============================================================
# TEST ONE REVIEW
# ============================================================

test_row = df_missing.iloc[0]

try:
    response = call_gemini(
        build_payload(test_row)
    )

    print("SUCCESS")
    print(
        "Overall sentiment:",
        response["result"]["overall_sentiment"]
    )

    print(
        "Features returned:",
        len(response["result"]["features"])
    )

    print(
        "Estimated cost:",
        f"${response['estimated_cost_usd']:.6f}"
    )

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))

In [ ]:
# ============================================================
# PROCESS THE REMAINING 12 REVIEWS
# ============================================================

recovery_stats = recover_gemini_sequential(
    df_missing
)

In [ ]:
# Identify the last missing review
completed = load_checkpoint()

df_last_missing = df_work.loc[
    [
        not is_current(
            completed.get(str(row["record_key"])),
            hash_text(build_payload(row))
        )
        for _, row in df_work.iterrows()
    ]
].copy()

print("Remaining reviews:", len(df_last_missing))

display(
    df_last_missing[
        ["record_key", "airline", "review_en"]
    ]
)

In [ ]:
from google.genai import types
import json

row = df_last_missing.iloc[0]
payload = build_payload(row)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=payload,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=RESPONSE_SCHEMA,
        max_output_tokens=8192,
        temperature=0,
        thinking_config=types.ThinkingConfig(
            thinking_level="minimal"
        )
    )
)

print("Response text present:", bool(response.text))

print("\nCandidate information:")
for candidate in response.candidates or []:
    print("Finish reason:", candidate.finish_reason)
    print("Safety ratings:", candidate.safety_ratings)

print("\nToken usage:")
print(response.usage_metadata)

if response.text:
    result = validate_prediction(
        json.loads(response.text)
    )
    print("\nSUCCESS — valid prediction returned.")
else:
    print("\nNo usable text returned. Inspect finish reason above.")

In [ ]:
print("Record key:", row["record_key"])

print("\nPrompt feedback:")
print(response.prompt_feedback)

print("\nCandidates:")
print(response.candidates)

print("\nReview length:", len(str(row["review_en"])))

print("\nReview text:")
print(str(row["review_en"])[:3000])

In [ ]:
# Export all successful Gemini classifications
df_gemini = export_gemini_results(df_work)

# Identify the blocked review
blocked_key = (
    "e2de67c99f8e9da4d76708f4db9a7cd00bdfd147f9ff81f87b2dbce126064877"
)

# Ensure all eligible reviews appear in the final dataset
missing_rows = df_work.loc[
    ~df_work["record_key"].astype(str).isin(
        df_gemini["record_key"].astype(str)
    )
].copy()

# Gemini columns should be missing for unclassified reviews
gemini_columns = [
    col for col in df_gemini.columns
    if col.endswith("_llm")
    or col.endswith("_sentiment_llm")
    or col.startswith("gemini_")
]

for col in gemini_columns:
    missing_rows[col] = pd.NA

missing_rows["gemini_status"] = "unclassified"

df_gemini["gemini_status"] = "completed"

# Combine classified and blocked reviews
df_gemini_final = pd.concat(
    [df_gemini, missing_rows],
    ignore_index=True
)

# Save
output_file = OUTPUT_DIR / "df_final_absa_gemini_complete.csv"

df_gemini_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Total reviews:", len(df_gemini_final))
print("Gemini classified:", (df_gemini_final["gemini_status"] == "completed").sum())
print("Gemini unclassified:", (df_gemini_final["gemini_status"] == "unclassified").sum())
print("Saved to:", output_file)